# Hyperparameter Tuning — LightGBM_v5

First of this version's two notebooks (the second is `02_training_LightGBM_v5.ipynb`). It:

1. **Picks this version's features:** a fixed list you choose, in `src/Training/04_Model-Development/LightGBM_v5/selected_features.xlsx` (`feature` column).
2. **Searches for good LightGBM hyperparameters** (the model's settings,
   e.g. tree size and learning rate) with Hyperopt: it tries 150
   combinations, scoring each by 5-fold cross-validated AUC on the
   training data, and keeps the best.

**Sections**
1. Settings
2. Load the data, pick the features, split off the test set
3. The ranges of hyperparameters to search
4. How each combination is scored
5. Run the search (or reuse the last one)
6. Save the results
7. Free memory

**Outputs** (`data/Model-Training/HyperParameters/LightGBM_v5/`)
- `selected_features.json` — the feature list; the training notebook reads it, so both always use the same features
- `lgbm_resolved_best.json` — the best hyperparameters found
- `lgbm_trials_history.csv` — every combination tried and its score

Run by the `04_model_pipeline` Airflow DAG (task `tune_model`), or by hand.

## 1. Imports & Configuration

`HYPERPARAMS_SOURCE` decides whether to search:

- `'search'` (default) — run the full search (can take hours) and save the best result.
- `'previous'` — skip the search and keep the hyperparameters saved last time
  (minutes). The feature list is still rebuilt and saved.

Under Airflow it's set by `04_model_pipeline`'s `tuning` option (through the
`HPO_HYPERPARAMS_SOURCE` environment variable). Two things aren't tuned
here: the number of trees and the yes/no threshold — the training notebook
chooses those itself with cross-validation.

`RANDOM_STATE` and `TEST_SIZE` must be the same here and in the training
notebook, so both hold back exactly the same 20% test set.

In [ ]:
import os
import gc; 
import json
from datetime import datetime
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from hyperopt import fmin, tpe, hp, Trials, space_eval

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")
HPO_DIR  = os.path.join(DATA_ROOT, "Model-Training", "HyperParameters", "LightGBM_v5")

os.makedirs(HPO_DIR, exist_ok=True)

PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_final.parquet")
PATH_PARS_BEST = os.path.join(HPO_DIR, "lgbm_resolved_best.json")
# This version's folder in src/, holding its fixed feature list (selected_features.xlsx)
PATH_SELECTED_VARS = os.path.join(SRC_ROOT, "Training", "04_Model-Development", "LightGBM_v5")
PATH_TRIALS_HISTORY = os.path.join(HPO_DIR, "lgbm_trials_history.csv")
PATH_FEATURE_LIST = os.path.join(HPO_DIR, "selected_features.json")  # read by the training notebook

# ── Train/test split -- MUST be the same in the training notebook ─────────
RANDOM_STATE = 1994
TEST_SIZE    = 0.20

# ── Whether to search ─────────────────────────────────────────────────────
#   'search'   -> run the search below and save the best result (default)
#   'previous' -> keep the hyperparameters saved last time (skips the search)
# Set by 04_model_pipeline's `tuning` option (HPO_HYPERPARAMS_SOURCE env var).
# The older HPO_USE_PREVIOUS_BEST=true still means 'previous'.
# (Training with the library's default hyperparameters is an option in the
# training notebook instead.)
HYPERPARAMS_SOURCE = os.environ.get("HPO_HYPERPARAMS_SOURCE", "").strip().lower()
if not HYPERPARAMS_SOURCE:
    HYPERPARAMS_SOURCE = ("previous" if os.environ.get("HPO_USE_PREVIOUS_BEST", "False").strip().lower()
                          in ("1", "true", "yes") else "search")
assert HYPERPARAMS_SOURCE in ("search", "previous"), f"unknown HYPERPARAMS_SOURCE {HYPERPARAMS_SOURCE!r}"
# How many hyperparameter combinations the search tries
MAX_EVALS = 150

In [ ]:
print(f"Hyperparameter source: {HYPERPARAMS_SOURCE}")

## 2. Load the Data, Pick the Features, Split Off the Test Set

The test set (20%) is split off with the same `RANDOM_STATE` and `TEST_SIZE`
as the training notebook, so the search never sees the applicants the final
model is evaluated on. Only the remaining 80% (`X_train`) is used below.

In [ ]:
model_dataset = pd.read_parquet(PATH_MODEL_DATASET)
# This version's features: the fixed list in its selected_features.xlsx.
# Saved to selected_features.json in section 6, which 02_training_LightGBM_v5.ipynb reads.
features_df = pd.read_excel(os.path.join(PATH_SELECTED_VARS, 'selected_features.xlsx'))
feature_cols = features_df['feature'].tolist()
print(f"Selected {len(feature_cols)} features (same as 02_training_LightGBM_v5.ipynb)")

# Make sure text columns are the 'category' type the model expects (parquet
# normally keeps it; this just guards against a file saved some other way)
cat_cols = model_dataset.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    model_dataset[col] = model_dataset[col].astype('category')

X = model_dataset[['SK_ID_CURR'] + feature_cols]
y = model_dataset['TARGET']
X.set_index('SK_ID_CURR', inplace=True)

print(f"X shape: {X.shape}")
print(f"Categorical columns: {len(cat_cols)}")

FEATURE_RULE = "fixed list: 04_Model-Development/LightGBM_v5/selected_features.xlsx"   # saved with the feature list

Shrink decimal columns from `float64` to `float32` to halve their memory use.

In [ ]:
float_cols = X.select_dtypes(include=['float64']).columns
X[float_cols] = X[float_cols].astype('float32')

In [ ]:
# Same split as the training notebook; X_val (the test set) isn't used by the search
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")

In [ ]:
# Free memory
del X, y, model_dataset
gc.collect()

## 3. The Ranges of Hyperparameters to Search

- **Tree shape:** `num_leaves` (leaves per tree), `max_depth`.
- **`learning_rate`:** how much each new tree corrects the previous ones.
- **`colsample_bytree`:** share of features each tree may use.
- **`reg_alpha`, `reg_lambda`:** penalties that keep the trees' outputs small (less overfitting).
- **`min_split_gain`:** a split must improve the fit by at least this much.
- **`min_child_samples`:** fewest training applicants allowed in a leaf.

Trees are built with `'goss'` boosting (Gradient-based One-Side Sampling),
which picks its own rows to learn from — so row sampling isn't searched.
The number of trees isn't searched either: each fit allows up to 500 and
stops early once more trees stop helping.

In [ ]:
BOOSTING_TYPE = 'goss'  # GOSS chooses its own rows, so row sampling isn't searched

space = {
    # Tree shape
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'max_depth':  hp.quniform('max_depth', 4, 12, 1),

    # How much each new tree corrects the previous ones (searched on a log scale)
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
  # 'n_estimators':  hp.quniform('n_estimators', 10, 1000, 10),

    # Share of features each tree may use
    'colsample_bytree': hp.uniform('colsample_bytree', 0.1, 1.0),

    # Penalties that keep the trees' outputs small
    'reg_alpha':  hp.loguniform('reg_alpha', np.log(1e-3), np.log(100.0)),
    'reg_lambda': hp.loguniform('reg_lambda', np.log(1e-3), np.log(100.0)),

    # A split is only made if it improves the fit by at least this much
    'min_split_gain': hp.loguniform('min_split_gain', np.log(1e-3), np.log(1.0)),

    # Fewest training applicants allowed in a leaf. The range is wide
    # (100–10,000): an earlier search with a 20–500 range kept picking 500,
    # the top of the range, which suggests the best value was higher. 20
    # (LightGBM's default) is also very small for ~250,000 training rows.
    'min_child_samples': hp.quniform('min_child_samples', 100, 10000, 20),
}

# These come out of Hyperopt as decimals and must be whole numbers
INT_PARAMS = ['num_leaves', 'max_depth', 'min_child_samples']

## 4. How Each Combination Is Scored

For one set of hyperparameters: 5-fold cross-validation on the training
data — train on 4/5, measure AUC on the other 1/5, five times — and take the
average AUC. Each fold stops adding trees once 20 in a row haven't improved
its AUC (early stopping). Hyperopt looks for the *lowest* score, so the
function returns minus the AUC.

A line per combination is printed (and shown live in the Airflow task log).

In [ ]:
def objective(params):
    """
    Score one set of hyperparameters: the average AUC over 5-fold
    cross-validation on the training data.

    Returns minus the AUC, because Hyperopt looks for the lowest value. The
    folds are looped over by hand (rather than with cross_val_score) so each
    fit can use LightGBM's early stopping.
    """
    current_params = params.copy()
    for p in INT_PARAMS:
        current_params[p] = int(current_params[p])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]   # 4/5: fit on these
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]       # 1/5: measure AUC on these

        model = lgb.LGBMClassifier(
            objective='binary',
            boosting_type=BOOSTING_TYPE,
            max_bin=63,           # fewer value buckets per feature -> faster
            device='cpu',
            seed=42,
            n_jobs=14,            # leave some CPU cores free for Airflow
            n_estimators=500,     # most trees allowed; early stopping usually ends sooner
            verbose=-1,
            **current_params,
        )
        model.fit(
            X_tr, y_tr,
            eval_metric='auc',
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=20,   # stop once 20 trees in a row haven't improved the AUC
                    verbose=False,
                    # first_metric_only=True,
                    # min_delta=0.001
                    )
                ],
        )

        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        # Free this fold's data before the next one
        del X_tr, y_tr, X_va, y_va, model
        gc.collect()


    mean_auc = np.mean(auc_scores)
    trial_aucs.append(mean_auc)
    # One line per combination -- also shown live in the Airflow task log
    print(f'[trial {len(trial_aucs)}/{MAX_EVALS}] CV AUC {mean_auc:.4f} | best so far {max(trial_aucs):.4f}', flush=True)
    return {'loss': -mean_auc, 'status': 'ok'}

## 5. Run the Search (or Reuse the Last One)

- `'previous'`: loads the hyperparameters saved last time.
- `'search'`: Hyperopt tries `MAX_EVALS` combinations. It uses TPE, which
  focuses later tries around the combinations that scored well so far.

Either way the result is one `resolved_best` dict, and the cell prints
which source it came from.

In [ ]:
if HYPERPARAMS_SOURCE == 'previous':
    print(f"HYPERPARAMS_SOURCE='previous' → loading saved params from {PATH_PARS_BEST}")
    with open(PATH_PARS_BEST) as f:
        resolved_best = json.load(f)
    trials = None

else:
    print(f"HYPERPARAMS_SOURCE='search' → running live Hyperopt search ({MAX_EVALS} evals)")
    trial_aucs = []  # each combination's average CV AUC, for the progress lines
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
        show_progressbar=False,  # objective() prints a line per combination instead
    )
    print("\n--- Hyperopt Search Complete ---")

    # Turn Hyperopt's raw result into actual parameter values, with whole numbers where needed
    resolved_best = space_eval(space, best)
    for p in INT_PARAMS:
        resolved_best[p] = int(resolved_best[p])

print(f"\nActive hyperparameter source: {HYPERPARAMS_SOURCE}")
print(json.dumps(resolved_best, indent=4) if resolved_best else "(none)")

## 6. Save the Results

- The best hyperparameters — only after a new search (`'previous'` leaves
  the saved file as it is).
- The search history (every combination and its AUC), as a CSV — useful to
  see whether the search was still improving.
- The feature list (`selected_features.json`), on **every** run, with the
  rule that produced it. The training notebook reads this rather than
  choosing features itself, so tuning and training always use the same list.

In [ ]:
if HYPERPARAMS_SOURCE == 'search':   # 'previous' leaves the saved file as is
    with open(PATH_PARS_BEST, 'w') as f:
        json.dump(resolved_best, f, indent=4)
        print(f"Saved resolved_best → {PATH_PARS_BEST}")

if trials is not None:
    # One row per combination tried: its AUC and its parameter values
    trials_df = pd.DataFrame(trials.results)
    trials_df['auc'] = -trials_df['loss']
    trials_df['trial_number'] = range(len(trials_df))

    params_df = pd.DataFrame([t['misc']['vals'] for t in trials.trials])
    params_df = params_df.applymap(lambda x: x[0] if isinstance(x, list) and x else None)

    trials_history = pd.concat([trials_df, params_df], axis=1)
    trials_history.to_csv(PATH_TRIALS_HISTORY, index=False)
    print(f"Saved trial history ({len(trials_history)} trials) → {PATH_TRIALS_HISTORY}")

    print(f"\nBest CV AUC found: {trials_history['auc'].max():.4f}")
else:
    print("No live search ran — trial history not available this run.")

# The feature list, saved on every run (search or previous). The training
# notebook reads this file, so both notebooks always use the same features.
feature_list = {
    'features': list(feature_cols),
    'rule': FEATURE_RULE,
    'hyperparameters_source': HYPERPARAMS_SOURCE,
    'created_at': datetime.now().isoformat(timespec='seconds'),
}
with open(PATH_FEATURE_LIST, 'w') as f:
    json.dump(feature_list, f, indent=2)
print(f"Saved feature list ({len(feature_cols)} features; {FEATURE_RULE}) → {PATH_FEATURE_LIST}")

## 7. Free Memory

In [ ]:
del X_train, X_val, y_train, y_val
gc.collect()
print("All DataFrames released.")